# AULA SPARK

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.appName('Aula de Spark').enableHiveSupport().getOrCreate()

In [4]:
spark

In [ ]:
#ler dados do hdfs

In [12]:
df_hdfs = spark.read.csv(path='/datalake/arquivo/',header=True)

In [13]:
df_hdfs.show()

+-------+----+---------+
|usuario|item|avaliacao|
+-------+----+---------+
|    109|   9|        3|
|    174| 412|        1|
|      7| 208|        5|
|    371|  97|        5|
|    296| 255|        2|
|    280|  82|        2|
|    271| 275|        4|
|    110| 791|        2|
|     59| 926|        1|
|    217| 576|        1|
|    145| 665|        5|
|    334| 204|        4|
|     42| 568|        4|
|    200| 143|        5|
|     89| 387|        5|
|    311| 588|        4|
|    235| 269|        4|
|    287| 156|        5|
|    344| 204|        4|
|     43| 289|        4|
+-------+----+---------+
only showing top 20 rows



In [ ]:
#ler dados do s3

In [14]:
df_s3 = spark.read.json('s3a://camada-bronze/city')

In [15]:
df_s3.show()

+---------+--------+----+--------------+-------+------------+-------+----------+----------------+
|__deleted|   __lsn|__op|__source_ts_ms|__table|        city|city_id|country_id|     last_update|
+---------+--------+----+--------------+-------+------------+-------+----------+----------------+
|    false|92662824|   c| 1678318110082|   city|Divinolandia|    652|        11|1678307310059665|
|    false|92662112|   c| 1678317762797|   city|  Araraquara|    651|        11|1678306962794627|
|    false|92685840|   c| 1678318113380|   city|    New York|    653|        11|1678307313380279|
|    false|92639248|   c| 1678317758356|   city|      Santos|    650|        11|1678306958332527|
+---------+--------+----+--------------+-------+------------+-------+----------+----------------+



In [16]:
#ler dados de banco de dados

In [17]:
conn = 'jdbc:postgresql://postgres:5432/dvdrental'

In [18]:
prop = {
    "user":"admin",
    "password": "admin",
    "driver": "org.postgresql.Driver"
    }

In [20]:
df_city = spark.read.jdbc(url=conn,properties=prop,table='public.city')

In [23]:
df_city.count()

652

In [28]:
query  = '(select c.customer_id,c.first_name, c.email ,c2.city from public.customer c \
inner join public.address a on c.address_id  = a.address_id \
inner join public.city c2 on c2.city_id  = a.city_id) as tab'

In [29]:
query

'(select c.customer_id,c.first_name, c.email ,c2.city from public.customer c inner join public.address a on c.address_id  = a.address_id inner join public.city c2 on c2.city_id  = a.city_id) as tab'

In [30]:
df_query = spark.read.jdbc(url=conn,properties=prop, table=query)

In [32]:
df_query.show(truncate=False)

+-----------+----------+-----------------------------------+---------------+
|customer_id|first_name|email                              |city           |
+-----------+----------+-----------------------------------+---------------+
|524        |Jared     |jared.ely@sakilacustomer.org       |Purwakarta     |
|1          |Mary      |mary.smith@sakilacustomer.org      |Sasebo         |
|2          |Patricia  |patricia.johnson@sakilacustomer.org|San Bernardino |
|3          |Linda     |linda.williams@sakilacustomer.org  |Athenai        |
|4          |Barbara   |barbara.jones@sakilacustomer.org   |Myingyan       |
|5          |Elizabeth |elizabeth.brown@sakilacustomer.org |Nantou         |
|6          |Jennifer  |jennifer.davis@sakilacustomer.org  |Laredo         |
|7          |Maria     |maria.miller@sakilacustomer.org    |Kragujevac     |
|8          |Susan     |susan.wilson@sakilacustomer.org    |Hamilton       |
|9          |Margaret  |margaret.moore@sakilacustomer.org  |Masqat         |

In [33]:
#salvo os dados

In [36]:
df_query.show(2)

+-----------+----------+--------------------+----------+
|customer_id|first_name|               email|      city|
+-----------+----------+--------------------+----------+
|        524|     Jared|jared.ely@sakilac...|Purwakarta|
|          1|      Mary|mary.smith@sakila...|    Sasebo|
+-----------+----------+--------------------+----------+
only showing top 2 rows



In [37]:
type(df_query)

pyspark.sql.dataframe.DataFrame

In [41]:
df_query.write.csv('hdfs:///datalake/query',header=True,sep=';')

AnalysisException: path hdfs://namenode:8020/datalake/query already exists.

In [40]:
df_query = df_query.repartition(3)

In [42]:
df_query.write.csv('hdfs:///datalake/query',header=True,sep=';',mode='overwrite')

In [43]:
df_query.write.json('s3a://camada-bronze/query')

In [44]:
df_query.write.parquet('s3a://camada-prata/query')

In [45]:
df_query.write.format('hive').saveAsTable('default.query')

In [49]:
df_query.write.mode('append').format('hive').insertInto('default.query')

In [50]:
#ler dados do hive

In [51]:
df_hive = spark.read.table('default.query')

In [52]:
df_hive.show()

+-----------+----------+--------------------+---------+
|customer_id|first_name|               email|     city|
+-----------+----------+--------------------+---------+
|        366|   Brandon|brandon.huey@saki...|Balikesir|
|        138|     Hazel|hazel.warren@saki...|   Hohhot|
|        255|      Irma|irma.pearson@saki...|  Hagonoy|
|        536|  Fernando|fernando.churchil...|  Tonghae|
|        520|  Mitchell|mitchell.westmore...|Nha Trang|
|        238|    Nellie|nellie.garrett@sa...|  Shimoga|
|        183|       Ida|ida.andrews@sakil...|  Luzinia|
|        438|     Barry|barry.lovelace@sa...|    Kitwe|
|        571|   Johnnie|johnnie.chisholm@...|    Plock|
|        579|     Daryl|daryl.larue@sakil...|    Mosul|
|        213|      Gina|gina.williamson@s...|    Taizz|
|         92|      Tina|tina.simmons@saki...|   Goinia|
|        356|    Gerald|gerald.fultz@saki...|    Satna|
|         73|   Beverly|beverly.brooks@sa...|   Chiayi|
|         43| Christine|christine.roberts...|   

In [53]:
df_hive.join(df_query,df_query.customer_id == df_hive.customer_id).show()

+-----------+----------+--------------------+--------------------+-----------+----------+--------------------+--------------------+
|customer_id|first_name|               email|                city|customer_id|first_name|               email|                city|
+-----------+----------+--------------------+--------------------+-----------+----------+--------------------+--------------------+
|        559|   Everett|everett.banda@sak...|             Bilbays|        559|   Everett|everett.banda@sak...|             Bilbays|
|        449|     Oscar|oscar.aquino@saki...|              Sirjan|        449|     Oscar|oscar.aquino@saki...|              Sirjan|
|        250|        Jo|jo.fowler@sakilac...|                 Oyo|        250|        Jo|jo.fowler@sakilac...|                 Oyo|
|         85|      Anne|anne.powell@sakil...|            Bradford|         85|      Anne|anne.powell@sakil...|            Bradford|
|        113|     Cindy|cindy.fisher@saki...|               Cuman|        11

In [54]:
#ler dados com SQL

In [60]:
df_total = spark.sql('select * from parquet.`s3a://camada-prata/query`')

In [61]:
df_total.show()

+-----------+----------+--------------------+---------+
|customer_id|first_name|               email|     city|
+-----------+----------+--------------------+---------+
|        366|   Brandon|brandon.huey@saki...|Balikesir|
|        138|     Hazel|hazel.warren@saki...|   Hohhot|
|        255|      Irma|irma.pearson@saki...|  Hagonoy|
|        536|  Fernando|fernando.churchil...|  Tonghae|
|        520|  Mitchell|mitchell.westmore...|Nha Trang|
|        238|    Nellie|nellie.garrett@sa...|  Shimoga|
|        183|       Ida|ida.andrews@sakil...|  Luzinia|
|        438|     Barry|barry.lovelace@sa...|    Kitwe|
|        571|   Johnnie|johnnie.chisholm@...|    Plock|
|        579|     Daryl|daryl.larue@sakil...|    Mosul|
|        213|      Gina|gina.williamson@s...|    Taizz|
|         92|      Tina|tina.simmons@saki...|   Goinia|
|        356|    Gerald|gerald.fultz@saki...|    Satna|
|         73|   Beverly|beverly.brooks@sa...|   Chiayi|
|         43| Christine|christine.roberts...|   

In [62]:
#escrever no banco de dados
df_total.write.jdbc(url=conn, properties=prop,table='public.aula')

In [63]:
#api

In [64]:
import requests

In [84]:
l = []
for x in range (10):
    print(x)
    retorno = requests.get('https://catfact.ninja/fact')
    l.append(retorno.json())

0
1
2
3
4
5
6
7
8
9


In [85]:
l

[{'fact': 'Foods that should not be given to cats include onions, garlic, green tomatoes, raw potatoes, chocolate, grapes, and raisins. Though milk is not toxic, it can cause an upset stomach and gas. Tylenol and aspirin are extremely toxic to cats, as are many common houseplants. Feeding cats dog food or canned tuna that’s for human consumption can cause malnutrition.',
  'length': 360},
 {'fact': 'The first formal cat show was held in England in 1871; in America, in 1895.',
  'length': 75},
 {'fact': 'Spanish-Jewish folklore recounts that Adam’s first wife, Lilith, became a black vampire cat, sucking the blood from sleeping babies. This may be the root of the superstition that a cat will smother a sleeping baby or suck out the child’s breath.',
  'length': 245},
 {'fact': 'Kittens who are taken along on short, trouble-free car trips to town tend to make good passengers when they get older. They get used to the sounds and motions of traveling and make less connection between the car a

In [67]:
retorno.status_code

200

{'fact': 'The term “puss” is the root of the principal word for “cat” in the Romanian term pisica and the root of secondary words in Lithuanian (puz) and Low German\xa0puus. Some scholars suggest that “puss” could be imitative of the hissing sound used to get a cat’s attention. As a slang word for the female pudenda, it could be associated with the connotation of a cat being soft, warm, and fuzzy.',
 'length': 387}

In [86]:
req = spark.read.json(spark.sparkContext.parallelize(l))

In [88]:
req.show()

+--------------------+------+
|                fact|length|
+--------------------+------+
|Foods that should...|   360|
|The first formal ...|    75|
|Spanish-Jewish fo...|   245|
|Kittens who are t...|   239|
|Cats should not b...|   196|
|Spanish-Jewish fo...|   245|
|The smallest wild...|   145|
|The cat appears t...|    84|
|A cat named Dusty...|   107|
|Cats must have fa...|    76|
+--------------------+------+



In [ ]:
req.repartition(1).write.json('s3a://camada-bronze/facts',mode='append')

In [93]:
df_teste1 = spark.read.parquet('/datalake/query/*parquet').show(1000)

+-----------+----------+--------------------+--------------------+
|customer_id|first_name|               email|                city|
+-----------+----------+--------------------+--------------------+
|        559|   Everett|everett.banda@sak...|             Bilbays|
|        449|     Oscar|oscar.aquino@saki...|              Sirjan|
|        250|        Jo|jo.fowler@sakilac...|                 Oyo|
|         85|      Anne|anne.powell@sakil...|            Bradford|
|        113|     Cindy|cindy.fisher@saki...|               Cuman|
|        293|       Mae|mae.fletcher@saki...|Donostia-San Seba...|
|        358|    Samuel|samuel.marlow@sak...|              Ranchi|
|        157|   Darlene|darlene.rose@saki...|           Pyongyang|
|        180|     Stacy|stacy.cunningham@...|             Pereira|
|        509|      Raul|raul.fortier@saki...|              Chapra|
|         77|      Jane|jane.bennett@saki...|            Araatuba|
|        349|       Joe|joe.gilliland@sak...|                I